# Tarea 4: genética, exposición ambiental y cáncer

**Consigna:** generar datos acordes al problema que se quiere estudiar y escribir las ecuaciones y el generador.

Pregunta: **¿cómo cambia la relación entre predisposición genética y exposición ambiental al estudiar solamente pacientes con cáncer?**

Construimos un ejemplo ficticio con tres variables binarias. Genética y exposición son independientes en la población, pero ambas aumentan el riesgo de cáncer. Al seleccionar solo pacientes con cáncer, aparecerá una asociación negativa entre estos dos factores.

Los números están elegidos para ilustrar el sesgo de selección por un colisionador; no son estimaciones médicas. La notebook usa NumPy, pandas y Matplotlib y se puede ejecutar de arriba hacia abajo sin descargar datos ni usar conexión a internet.

## 1. La historia y las variables

Imaginemos una población donde algunas personas tienen predisposición genética fuerte y otras tienen exposición ambiental fuerte. La exposición podría representar contacto con un contaminante. En este modelo, ambos factores aumentan la probabilidad de cáncer.

| Símbolo | Variable | Valores |
|---|---|---|
| $G$ | Predisposición genética fuerte | 1: sí; 0: no |
| $E$ | Exposición ambiental fuerte | 1: sí; 0: no |
| $C$ | Cáncer | 1: tiene cáncer; 0: no tiene cáncer |

Generaremos $G$ y $E$ de forma independiente: tener predisposición genética no cambia la probabilidad de exposición ambiental en la población del ejemplo. Supondremos también que no hay causas comunes adicionales de esos dos factores.

Compararemos **toda la población** con la muestra formada únicamente por **pacientes con cáncer**, es decir, el subconjunto con $C=1$. Observamos las tres variables; no necesitamos resultados latentes ni datos faltantes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import display

plt.rcParams['figure.dpi'] = 110

## 2. El DAG: un efecto común

$$G \longrightarrow C \longleftarrow E$$

$G$ y $E$ son causas de $C$, pero ninguno causa al otro. Cáncer es un **colisionador** porque recibe las dos flechas.

En la población, el camino entre $G$ y $E$ está bloqueado por el colisionador: $G \perp E$. Seleccionar solo pacientes con cáncer equivale a condicionar en $C=1$ y abre ese camino. Bajo las probabilidades que elegiremos, $G$ y $E$ quedarán asociados negativamente en la muestra seleccionada.

El DAG indica que condicionar puede inducir asociación; su signo y magnitud dependen de las ecuaciones generadoras.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
positions = {'G': (0, 1.5), 'E': (3, 1.5), 'C': (1.5, 0)}
labels = {
    'G': 'G: Strong genetic\npredisposition',
    'E': 'E: Strong environmental\nexposure',
    'C': 'C: Cancer\nSelect C = 1',
}
colors = {'G': '#dbeafe', 'E': '#dcfce7', 'C': '#fee2e2'}

for start, end in [((0.4, 1.1), (1.1, 0.4)), ((2.6, 1.1), (1.9, 0.4))]:
    ax.annotate(
        '', xy=end, xytext=start,
        arrowprops={'arrowstyle': '-|>', 'color': '#374151', 'lw': 2,
                    'mutation_scale': 18},
    )

for node, (x, y) in positions.items():
    patch = FancyBboxPatch(
        (x - 0.85, y - 0.30), 1.70, 0.60,
        boxstyle='round,pad=0.05', facecolor=colors[node],
        edgecolor='#dc2626' if node == 'C' else '#374151',
        linewidth=2.5 if node == 'C' else 1.2,
    )
    ax.add_patch(patch)
    ax.text(x, y, labels[node], ha='center', va='center', fontsize=10)

ax.set_xlim(-1.1, 4.1)
ax.set_ylim(-0.6, 2.1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('G and E share a collider: select cancer patients (C = 1)')
fig.tight_layout()
plt.show()

## 3. Las ecuaciones generadoras

Para cada persona $i$, tomamos tres errores independientes entre sí y entre personas:

$$\varepsilon_i^G,\varepsilon_i^E,\varepsilon_i^C
\overset{\mathrm{iid}}{\sim}\operatorname{Uniforme}(0,1).$$

Si $\mathbf{1}\{\cdot\}$ vale 1 cuando se cumple la condición y 0 en caso contrario:

$$G_i=\mathbf{1}\{\varepsilon_i^G<0.50\},\qquad
E_i=\mathbf{1}\{\varepsilon_i^E<0.50\}.$$

$$p_C(G_i,E_i)=0.10+0.40G_i+0.40E_i,$$

$$C_i=\mathbf{1}\{\varepsilon_i^C<p_C(G_i,E_i)\}.$$

Por lo tanto, $G$ y $E$ son Bernoulli independientes con probabilidad 0.50. El cáncer es Bernoulli condicional a ambos factores:

| $G$ | $E$ | $P(C=1\mid G,E)$ |
|---|---|---|
| 0 | 0 | 0.10 |
| 0 | 1 | 0.50 |
| 1 | 0 | 0.50 |
| 1 | 1 | 0.90 |

Cada factor aumenta la probabilidad de cáncer en **40 puntos porcentuales**, manteniendo fijo el otro. Todas las probabilidades están entre 0 y 1; no necesitamos funciones logísticas ni recortes.

In [ ]:
def generate_data(n=10_000, seed=2026):
    rng = np.random.default_rng(seed)
    genetic_predisposition = (rng.uniform(size=n) < 0.50).astype(int)
    environmental_exposure = (rng.uniform(size=n) < 0.50).astype(int)
    cancer_probability = (
        0.10 + 0.40 * genetic_predisposition + 0.40 * environmental_exposure
    )
    cancer = (rng.uniform(size=n) < cancer_probability).astype(int)

    return pd.DataFrame({
        'G': genetic_predisposition,
        'E': environmental_exposure,
        'C': cancer,
    })


data = generate_data()
cancer_patients = data.loc[data['C'] == 1].copy()

## 4. Los datos generados

La primera tabla contiene las primeras personas de la población simulada. La segunda muestra únicamente pacientes con cáncer: allí $C=1$ para todos, pero $G$ y $E$ pueden variar.

In [ ]:
print('Full population:')
display(data.head(10))
print('Cancer patients only:')
display(cancer_patients.head(10))
print(f'Total population: {len(data):,}')
print(f'Cancer patients: {len(cancer_patients):,}')

## 5. La relación entre genética y exposición

Comparamos la proporción de personas expuestas ($E=1$) entre quienes tienen y no tienen predisposición genética fuerte ($G=1$ y $G=0$). Lo hacemos primero en la población y luego entre pacientes con cáncer.

En la población esperamos proporciones cercanas al 50% en ambos grupos. Entre pacientes con cáncer esperamos mayor exposición ambiental en el grupo sin predisposición genética fuerte.

La diferencia se calcula como proporción expuesta en $G=1$ **menos** proporción expuesta en $G=0$, expresada en puntos porcentuales. Una diferencia negativa indica asociación negativa entre $G$ y $E$; no significa que un factor reduzca causalmente al otro.

In [ ]:
rows = []
for sample_name, frame in [
    ('Full population', data),
    ('Cancer patients only', cancer_patients),
]:
    for genetics in [0, 1]:
        group = frame.loc[frame['G'] == genetics]
        rows.append({
            'sample': sample_name,
            'G': genetics,
            'n': len(group),
            'exposure_pct': 100 * group['E'].mean(),
        })

exposure_summary = pd.DataFrame(rows)
exposure_by_group = exposure_summary.pivot(
    index='sample', columns='G', values='exposure_pct'
).reindex(['Full population', 'Cancer patients only'])
exposure_difference = exposure_by_group[1] - exposure_by_group[0]

display(exposure_summary.round(2))
display(exposure_difference.rename('exposure_difference_pp').to_frame().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, sample_name in zip(axes, exposure_by_group.index):
    values = exposure_by_group.loc[sample_name, [0, 1]].to_numpy()
    bars = ax.bar(['G = 0', 'G = 1'], values, color=['#2563eb', '#ea580c'])
    ax.bar_label(bars, labels=[f'{value:.1f}%' for value in values], padding=3)
    ax.set_title(
        f'{sample_name}\nDifference: {exposure_difference.loc[sample_name]:+.1f} pp'
    )
    ax.set_xlabel('Strong genetic predisposition')
    ax.set_ylabel('Strong environmental exposure (%)')
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)

fig.suptitle('Selecting cancer patients induces an association between G and E')
fig.tight_layout()
plt.show()

### Scatter: antes y después de seleccionar

Cada punto representa una persona, con $G$ en el eje horizontal y $E$ en el vertical. Como ambas variables valen 0 o 1, agregamos un pequeño desplazamiento aleatorio (*jitter*) **solo para dibujar**: los datos siguen siendo binarios y no cambian.

Comparamos la población completa con sus pacientes con cáncer usando los mismos desplazamientos para cada persona. La línea une la proporción real de personas expuestas en $G=0$ y $G=1$, calculada sin jitter. En la población debería ser casi horizontal; entre pacientes con cáncer debería bajar. Los títulos muestran la correlación calculada con los datos binarios originales.

Las cuatro nubes corresponden a las cuatro combinaciones posibles de factores. La línea y sus etiquetas ayudan a ver la asociación negativa que surge al seleccionar; su pendiente describe una asociación, no un efecto causal de $G$ sobre $E$.

In [ ]:
jitter_rng = np.random.default_rng(2027)
plot_coordinates = pd.DataFrame({
    'G_plot': data['G'] + jitter_rng.uniform(-0.10, 0.10, len(data)),
    'E_plot': data['E'] + jitter_rng.uniform(-0.10, 0.10, len(data)),
}, index=data.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, sample_name, frame, color in [
    (axes[0], 'Full population', data, '#2563eb'),
    (axes[1], 'Cancer patients only (C = 1)', cancer_patients, '#dc2626'),
]:
    coordinates = plot_coordinates.loc[frame.index]
    ax.scatter(
        coordinates['G_plot'], coordinates['E_plot'],
        s=8, alpha=0.07, color=color, edgecolors='none', rasterized=True,
    )
    exposure_means = frame.groupby('G')['E'].mean().reindex([0, 1])
    ax.plot(
        [0, 1], exposure_means.to_numpy(),
        color='#111827', linewidth=2.5, marker='D', markersize=7,
        label='Mean E by G (original data)', zorder=3,
    )
    for genetics, mean_exposure in exposure_means.items():
        ax.annotate(
            f'{100 * mean_exposure:.1f}% exposed',
            xy=(genetics, mean_exposure), xytext=(0, 12),
            textcoords='offset points', ha='center', fontsize=10,
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.9},
            zorder=4,
        )
    correlation = frame['G'].corr(frame['E'])
    ax.set_title(f'{sample_name}\nn = {len(frame):,}; corr(G, E) = {correlation:+.3f}')
    ax.set_xlabel('Strong genetic predisposition (G)')
    ax.set_ylabel('Strong environmental exposure (E)')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlim(-0.25, 1.25)
    ax.set_ylim(-0.25, 1.25)
    ax.grid(alpha=0.15)
    ax.set_axisbelow(True)
    ax.legend(loc='lower center', fontsize=9)

fig.suptitle('Selection bias: independent factors become associated among cancer patients')
fig.tight_layout()
plt.show()

## 6. Ambos factores siguen aumentando el riesgo de cáncer

Para ver la relación de los factores con el cáncer usamos **la población completa**, donde hay personas con y sin cáncer. Mostramos el riesgo de cáncer en cada combinación de $G$ y $E$ y lo comparamos con las probabilidades del generador.

Entre pacientes con cáncer no podríamos hacer esta comparación: el resultado $C$ sería constante. La asociación negativa entre genética y exposición en esa muestra no implica que alguno de los factores proteja contra el cáncer.

In [ ]:
cancer_risk_summary = data.groupby(['G', 'E'])['C'].agg(
    n='size', cancer_risk='mean'
).reset_index()
cancer_risk_summary['cancer_risk_pct'] = 100 * cancer_risk_summary['cancer_risk']
cancer_risk_summary['generator_risk_pct'] = 100 * (
    0.10 + 0.40 * cancer_risk_summary['G'] + 0.40 * cancer_risk_summary['E']
)
display(cancer_risk_summary[['G', 'E', 'n', 'cancer_risk_pct', 'generator_risk_pct']].round(2))

group_labels = [
    f'G = {genetics}, E = {exposure}'
    for genetics, exposure in zip(cancer_risk_summary['G'], cancer_risk_summary['E'])
]
x = np.arange(len(cancer_risk_summary))
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(x, cancer_risk_summary['cancer_risk_pct'], color='#7c3aed', label='Simulation')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.scatter(
    x, cancer_risk_summary['generator_risk_pct'],
    color='black', marker='_', s=350, linewidths=2, label='Generator', zorder=3,
)
ax.set_xticks(x, group_labels)
ax.set_ylim(0, 105)
ax.set_ylabel('Cancer risk (%)')
ax.set_title('Both factors increase cancer risk in the full population')
ax.legend()
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 7. La asociación inducida también se puede calcular

Por independencia, en toda la población:

$$P(E=1\mid G=0)=P(E=1\mid G=1)=0.50.$$

Al seleccionar pacientes con cáncer, aplicamos Bayes. Como $E$ es binaria e independiente de $G$:

$$P(E=1\mid G=g,C=1)=
\frac{P(C=1\mid G=g,E=1)P(E=1)}
{\sum_{e\in\{0,1\}}P(C=1\mid G=g,E=e)P(E=e)}.$$

Para personas sin predisposición genética fuerte:

$$P(E=1\mid G=0,C=1)=
\frac{0.50(0.50)}{0.10(0.50)+0.50(0.50)}
=\frac{5}{6}\approx0.833.$$

Para personas con predisposición genética fuerte:

$$P(E=1\mid G=1,C=1)=
\frac{0.90(0.50)}{0.50(0.50)+0.90(0.50)}
=\frac{9}{14}\approx0.643.$$

La diferencia esperada pasa de **0 puntos porcentuales** en la población a aproximadamente **−19.05 puntos porcentuales** entre pacientes con cáncer. La simulación debería dar valores cercanos, con pequeñas diferencias por variación muestral.

Intuitivamente, en la muestra seleccionada, las personas sin predisposición genética fuerte están más enriquecidas en exposición ambiental fuerte. Eso es un cambio en la composición de los grupos; no significa que cada caso tenga una única causa ni que los dos factores se excluyan.

## Conclusión

$G$ y $E$ son independientes por construcción. Ambos aumentan el riesgo de cáncer. Sin embargo, restringir el análisis a pacientes con cáncer abre el camino $G\to C\leftarrow E$ y genera una asociación negativa entre las causas del colisionador.

**La asociación en una muestra seleccionada puede surgir exclusivamente por la selección.** En este ejemplo, no permite concluir que la genética reduzca la exposición ni que la exposición reduzca el riesgo de cáncer.

**Material de referencia:** `causal_diagrams_s03_ss02_rc_v1jw-en.txt` menciona los determinantes genéticos y ambientales del cáncer como ejemplo de sesgo por colisionador. La estructura didáctica sigue la notebook `../tarea3.ipynb`.